---
## IMPORTS
---

In [ ]:
from __future__ import annotations

import json
from collections.abc import Callable
from dataclasses import KW_ONLY, dataclass
from datetime import datetime, timedelta
from typing import Any, NotRequired, Required, TypedDict

from anthropic.types import (
    Message,
    MessageParam,
    TextBlock,
    ToolChoiceToolParam,
    ToolParam,
    ToolResultBlockParam,
    ToolUseBlock,
)
from pydantic import BaseModel, Json, ValidationError
from rich.console import Console
from rich.markdown import Markdown

from learning_journey.claude.client import get_client

In [ ]:
class LLMParams(TypedDict, total=False):
    model: Required[str]
    max_tokens: Required[int]
    messages: Required[list[MessageParam]]
    system: NotRequired[str]
    stop_sequences: NotRequired[list[str]]
    tools: NotRequired[list[ToolParam]]
    tool_choice: NotRequired[ToolChoiceToolParam]


@dataclass(frozen=True, slots=True)
class ToolsInfo:
    func: Callable[..., str]
    _: KW_ONLY
    name: str
    schema: ToolParam

---
#### Batch tool with Pydantic v2: validate Claude's batch_tool input, then
#### run each inner tool call. Paste as one notebook cell; it runs end-to-end.
---

In [ ]:
# ---------------------------------------------------------------------
# 1. MODELS: mirror batch_tool_schema["input_schema"] one-to-one
# ---------------------------------------------------------------------
class Invocation(BaseModel):
    """One inner tool call inside the batch (one item of the array)."""

    name: str
    # ^ Tool to run, e.g. "get_weather". Must be a key in TOOL_REGISTRY.

    arguments: Json[dict[str, Any]]
    # ^ Claude sends this as a JSON *string*: '{"location": "SF"}'
    #   Json[...] does two steps for you:
    #     1) json.loads() the string          -> fails if not valid JSON
    #     2) validate the result as dict[...] -> fails if it's a list/number
    #   After validation, inv.arguments is a real dict: {"location": "SF"}
    #   Note: a dict passed directly (not a string) FAILS. Json[] only
    #   accepts str / bytes / bytearray.


class BatchInput(BaseModel):
    """The whole input object Claude sends to batch_tool."""

    invocations: list[Invocation]
    # ^ Each element is validated as an Invocation. If ANY item is bad,
    #   model_validate raises ONE ValidationError listing every problem
    #   with its exact location, e.g. ('invocations', 0, 'arguments').

---
## REGISTER TOOL FUNCTIONS
---

In [ ]:
type RegisterFunc = Callable[[Callable[..., str]], Callable[..., str]]

tools: dict[str, ToolsInfo] = {}


def register_func(name: str, schema: ToolParam) -> RegisterFunc:

    def decorator(func: Callable[..., str]) -> Callable[..., str]:
        tools[name] = ToolsInfo(
            func,
            name=name,
            schema=schema,
        )
        return func

    return decorator

---
## CLIENT INITIALIZATION AND HELPER FUNCTIONS
---

In [ ]:
client, settings = get_client()


def add_user_message(
    messages: list[MessageParam],
    message: Message | list[ToolResultBlockParam] | str,
) -> None:
    user_message: MessageParam = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], message: Message | str) -> None:
    assistant_message: MessageParam = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def add_tool_result(
    results: list[ToolResultBlockParam], *, tool_id: str, func_result: Any, is_error: bool = False
) -> None:
    results.append(
        ToolResultBlockParam(
            tool_use_id=tool_id,
            type="tool_result",
            content=func_result,
            is_error=is_error,
        )
    )


def chat(
    messages: list[MessageParam],
    *,
    system: str | None = None,
    stop_sequences: list[str] | None = None,
    tools: list[ToolParam] | None = None,
    tool_choice: ToolChoiceToolParam | None = None,
) -> Message:
    params: LLMParams = {
        "model": settings.claude_model,
        "max_tokens": settings.default_max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tools:
        params["tools"] = tools
    if tool_choice:
        params["tool_choice"] = tool_choice

    # Console().print(params)
    return client.messages.create(**params)

In [ ]:
def text_from_message(message: Message) -> str:
    return "\n".join([block.text for block in message.content if isinstance(block, TextBlock)])


def _tools_from_message(message: Message) -> list[ToolUseBlock]:
    return [block for block in message.content if isinstance(block, ToolUseBlock)]


def _process_tool_call(name: str, arguments: dict[str, Any]) -> str:
    """Run one tool by name. arguments is already a dict (parsed by Pydantic)."""
    if name not in tools:
        raise ValueError(f"Unknown tool: {name}")
    return tools[name].func(**arguments)
    # ^ ** unpacks the dict into keyword args:
    #   {"location": "SF"} -> get_weather(location="SF")


def run_tools(response: Message, results: list[ToolResultBlockParam]) -> list[ToolResultBlockParam]:
    tool_requests = _tools_from_message(response)

    for tool in tool_requests:
        try:
            result = _process_tool_call(tool.name, tool.input)
            add_tool_result(
                results,
                tool_id=tool.id,
                func_result=json.dumps(result),
                is_error=False,
            )
        except (ValidationError, Exception) as exc:
            exc_content = exc.json(include_url=False) if isinstance(exc, ValidationError) else exc
            add_tool_result(
                results, tool_id=tool.id, func_result=f"Error: {exc_content} ", is_error=True
            )
    return results

In [ ]:
# Tools and Schemas
add_duration_to_datetime_schema: ToolParam = {
    "name": "add_duration_to_datetime",
    "description": """Adds a specified duration to a datetime string and returns the resulting
    datetime in a detailed format. This tool converts an input datetime string to a Python datetime
    object, adds the specified duration in the requested unit, and returns a formatted string of
    the resulting datetime. It handles various time units including seconds, minutes, hours,
    days, weeks, months, and years, with special handling for month and year calculations to
    account for varying month lengths and leap years. The output is always returned in a detailed
    format that includes the day of the week, month name, day, year, and time with AM/PM indicator
    (e.g., 'Thursday, April 03, 2025 10:30:00 AM').
    """,
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": """The input datetime string to which the duration will be added.
                This should be formatted according to the input_format parameter.
                """,
            },
            "duration": {
                "type": "number",
                "description": """The amount of time to add to the datetime. Can be positive
                (for future dates) or negative (for past dates). Defaults to 0.
                """,
            },
            "unit": {
                "type": "string",
                "description": """The unit of time for the duration. Must be one of: 'seconds',
                'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.
                """,
            },
            "input_format": {
                "type": "string",
                "description": """The format string for parsing the input datetime_str, using
                Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates
                like '2025-04-03'. Defaults to '%Y-%m-%d'.
                """,
            },
        },
        "required": ["datetime_str"],
    },
}


@register_func("add_duration_to_datetime", add_duration_to_datetime_schema)
def add_duration_to_datetime(
    datetime_str: str, duration: int = 0, unit: str = "days", input_format: str = "%Y-%m-%d"
) -> str:
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


set_reminder_schema: ToolParam = {
    "name": "set_reminder",
    "description": """Creates a timed reminder that will notify the user at the specified time with
    the provided content. This tool schedules a notification to be delivered to the user at the
    exact timestamp provided. It should be used when a user wants to be reminded about something
    specific at a future point in time. The reminder system will store the content and timestamp,
    then trigger a notification through the user's preferred notification channels (mobile alerts,
    email, etc.) when the specified time arrives. Reminders are persisted even if the application
    is closed or the device is restarted. Users can rely on this function for important
    time-sensitive notifications such as meetings, tasks, medication schedules, or any
    other time-bound activities.
    """,
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": """The message text that will be displayed in the reminder
                notification. This should contain the specific information the user wants
                to be reminded about, such as 'Take medication', 'Join video call with team',
                or 'Pay utility bills'.
                """,
            },
            "timestamp": {
                "type": "string",
                "description": """The exact date and time when the reminder should be triggered,
                formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp.
                The system handles all timezone processing internally, ensuring reminders are
                triggered at the correct time regardless of where the user is located. Users can
                simply specify the desired time without worrying about timezone configurations.
                """,
            },
        },
        "required": ["content", "timestamp"],
    },
}


@register_func("set_reminder", set_reminder_schema)
def set_reminder(content: str, timestamp: str) -> str:
    return f"Reminder set for {timestamp}:\n{content}"


batch_tool_schema: ToolParam = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


# Takes validated MODELS, not raw dicts
@register_func("batch_tool", batch_tool_schema)
def batch_tool(invocations: list[Invocation]) -> str:
    batch_results: list[dict[str, str]] = []
    raised = 0
    for inv in invocations:
        try:
            parsed = Invocation.model_validate(inv)
            output = _process_tool_call(parsed.name, parsed.arguments)
            batch_results.append({"name": parsed.name, "result": output})
        except ValidationError as exc:
            batch_results.append({"name": inv.name, "error:": exc.json(include_url=False)})
            raised += 1
        except Exception as exc:
            batch_results.append({"name": inv.name, "error": str(exc)})
            raised += 1

    if raised > 0:
        raise Exception(json.dumps(batch_results))
    return json.dumps(batch_results)

In [ ]:
# Ask claude for the schema so you are not manually creating it.
get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": """
            Returns the current date and time as a string, using the local clock and timezone of the
            server that runs the tool. The value is timezone-aware, and the default output includes
            the timezone name. Use this tool whenever an answer depends on the present moment:
            today's date, the current time, the day of the week, or as the starting point for
            calculating how long it is until or since a date. Do not use it to convert or
            parse dates the user has already given, or to do date arithmetic; it only reads
            the current time.
            The output is formatted with a Python strftime format string. If date_format is omitted,
            the default '%Y-%m-%d %H:%M:%S %Z' is used (for example, '2026-11-24 16:22:05 EST').
            The server's timezone may differ from the user's, so check the timezone in the result
            before you tell the user the time. An empty date_format is rejected with an error.
            """,
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": """
                        A Python strftime format string that controls how the result looks.
                        Common directives: %Y = 4-digit year, %m = month (01-12), %d = day (01-31),
                        %H = hour (00-23), %I = hour (01-12), %M = minute, %S = second, %p = AM/PM,
                        %A = full weekday name, %B = full month name, %j = day of year,
                        %Z = timezone name (e.g. 'EST'), %z = UTC offset (e.g. '-0500').
                        Examples: '%Y-%m-%d' gives '2026-11-24'; '%A, %B %d, %Y' gives
                        'Tuesday, November 24, 2026'; '%I:%M %p %Z' gives '04:22 PM EST';
                        '%Y-%m-%dT%H:%M:%S%z' gives '2026-11-24T16:22:05-0500'. Leave this out to
                        get the default '%Y-%m-%d %H:%M:%S %Z'. Must not be an empty string.
                        """,
                    "default": "%Y-%m-%d %H:%M:%S %Z",
                    "minLength": 1,
                }
            },
            "required": [],
            "additionalProperties": False,
        },
        "input_examples": [{}, {"date_format": "%Y-%m-%d"}, {"date_format": "%Y-%m-%dT%H:%M:%S%z"}],
    }
)


@register_func("get_current_datetime", get_current_datetime_schema)
def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S %Z") -> str:
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().astimezone().strftime(date_format)

In [ ]:
def run_conversation(messages: list[MessageParam], **chat_settings: Any) -> Message:
    console = Console()

    while True:
        results: list[ToolResultBlockParam] = []
        # console.print(messages)
        response = chat(messages, **chat_settings)
        # console.print(response)

        add_assistant_message(messages, response)

        if text := text_from_message(response):
            # code_theme= changes the colors of code blocks.
            console.print(Markdown(text, code_theme="monokai"))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response, results)
        add_user_message(messages, tool_results)
    return response

In [ ]:
console = Console()
list_tools: list[ToolParam] = [tool_schema.schema for tool_schema in tools.values()]
tool_choice: ToolChoiceToolParam = {
    "name": "batch_tool",
    "type": "tool",
}

messages: list[MessageParam] = []
prompt: str = """
Set a reminder for my doctors appt. Its 177 days from now, at 3 pm my time zone.
"""

add_user_message(messages, prompt)
run_conversation(messages, tools=list_tools)

# To inspect the raw dicts while debugging, use console.print_json(data=r) or console.print(r).
# Both print them with colors and indentation, and Markdown rendering would hide the structure.
# console.print(tools)
console.print(messages)